# Chapter 8 · Function calling

Three tools for the North Cycles assistant (order status, stock and shipping cost), described with JSON schemas, and the full loop: the model asks for a tool, the program runs it and the model answers with the result. It is measured with twenty requests and four variants of the system message and the descriptions.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model and the data

In [ ]:
"""Chapter 8: function calling. Open model, on GPU or CPU."""
import json
import re
import unicodedata
import urllib.request
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-0.6B"             # without a GPU: smaller and faster
MODEL = None                              # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"



def load(name=None):
    """Loads the model and its tokenizer on the GPU if there is one; otherwise, CPU."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if name is None:
        name = GPU_MODEL if device == "cuda" else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    model.to(device).eval()
    print(f"Model {name} on {device}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)
ORDERS, WAREHOUSE = download("orders.json"), download("stock.json")
print(len(ORDERS), "orders and", len(WAREHOUSE), "products in the warehouse")

## 2. The tools: ordinary Python functions

They can be tried without any model.

In [ ]:
def strip_accents(text):
    """Lowercase and without accents."""
    return "".join(c for c in unicodedata.normalize("NFD", str(text).lower())
                   if unicodedata.category(c) != "Mn").strip()


def order_status(order_number):
    """Status of an order in the database."""
    digits = re.sub(r"\D", "", str(order_number))
    order = ORDERS.get(f"NC-{digits}")
    if order is None:
        return {"error": f"There is no order with the number {order_number}."}
    return {"order_number": f"NC-{digits}", **order}


def stock(product, size=None):
    """Units of a product in the warehouse, by name or by reference."""
    wanted = strip_accents(product)
    for ref, data in WAREHOUSE.items():
        if wanted in (strip_accents(ref), strip_accents(data["name"])):
            units = data["units"]
            if size is not None:
                units = {str(size): units.get(str(size).upper(), 0)}
            return {"reference": ref, "name": data["name"],
                    "units": units,
                    "restock": data.get("restock", "no date planned")}
    return {"error": f"There is no stock data for “{product}”."}


# The first zone that matches wins, so US places whose names contain a word of
# another zone (Vancouver, Washington; New Mexico; Paris, Texas) come first.
ZONES = {"contiguous US": ["new mexico", "washington", "vancouver, wa", "texas",
                           "california"],
         "Alaska": ["alaska", "anchorage", "fairbanks", "juneau"],
         "Hawaii": ["hawaii", "honolulu", "maui", "oahu", "hilo"],
         "Canada": ["canada", "toronto", "vancouver", "montreal"],
         "Puerto Rico": ["puerto rico", "san juan"],
         "abroad": ["france", "italy", "germany", "united kingdom", "spain",
                    "mexico", "japan", "paris", "london", "tokyo"]}


def shipping_cost(destination, amount=0):
    """Cost and delivery time of an order according to its destination and amount."""
    place = strip_accents(destination)
    zone = next((z for z, places in ZONES.items()
                 if any(p in place for p in places)), "contiguous US")
    if zone == "abroad":
        return {"error": f"We don't ship to {destination}. A quote can be requested "
                         "at hello@northcycles.com."}
    if zone == "contiguous US":
        cost = 0 if float(amount or 0) > 50 else 4.95
        return {"zone": zone, "cost_dollars": cost, "time": "2-3 business days"}
    rates = {"Alaska": (9.95, "3-5 business days"),
             "Hawaii": (14.95, "5-7 business days"),
             "Canada": (12.95, "3-5 business days"),
             "Puerto Rico": (12.95, "3-5 business days")}
    cost, time = rates[zone]
    return {"zone": zone, "cost_dollars": cost, "time": time}


FUNCTIONS = {"order_status": order_status, "stock": stock,
             "shipping_cost": shipping_cost}

In [ ]:
print(order_status("nc 10871"))
print(stock("Aero 3", "54"))
print(shipping_cost("Anchorage", 80))
print(shipping_cost("France"))

## 3. Describing the tools to the model

In [ ]:
def tool(name, description, parameters, required):
    """Schema of a tool in the format that models understand."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": parameters,
                       "required": required}}}


SCHEMAS = [
    tool(
        "order_status",
        "Looks up the status of a store order: whether it is processing, "
        "shipped, delivered, delayed or canceled, with its dates and its "
        "tracking number. Use it only if the customer gives their order number.",
        {"order_number": {"type": "string",
                          "description": "Order number, in the format NC-12345."}},
        ["order_number"]),
    tool(
        "stock",
        "Looks up how many units of a product are left in the warehouse, by "
        "size, and the restock date if it is out of stock.",
        {"product": {"type": "string",
                     "description": "Name of the product (for example, Aero 3) "
                                    "or its reference (for example, CR-7010)."},
         "size": {"type": "string",
                  "description": "Size, if the customer gives one (S, M, L, 54…)."}},
        ["product"]),
    tool(
        "shipping_cost",
        "Calculates the shipping cost and delivery time of an order according "
        "to the destination and the order amount.",
        {"destination": {"type": "string",
                         "description": "Destination city, state or country."},
         "amount": {"type": "number",
                    "description": "Order amount in dollars; 0 if unknown."}},
        ["destination"]),
]


# The same tools, without descriptions: only names and types.
BARE_SCHEMAS = [
    tool(s["function"]["name"], "",
         {p: {"type": d["type"]}
          for p, d in s["function"]["parameters"]["properties"].items()},
         s["function"]["parameters"]["required"]) for s in SCHEMAS]


SYSTEM_EN = (
    "You are the customer service assistant of North Cycles, a bicycle store "
    "in Portland, Oregon. Answer in English, in one or two sentences. For "
    "order, stock and shipping data, use the tools; do not make them up. If "
    "you are missing a piece of information to use a tool, ask the customer "
    "for it. Other store information: customers have 30 days to return a "
    "product; the warranty is 2 years; the store is open Monday to Friday "
    "from 10 a.m. to 8 p.m. and Saturdays from 10 a.m. to 2 p.m.")


# The same instructions in Spanish, a foreign language for this store, to
# see whether the language of the system message changes the use of tools.
SYSTEM_ES = (
    "Eres el asistente de atención al cliente de North Cycles, una tienda de "
    "bicicletas de Portland (Oregón, Estados Unidos). Responde siempre en "
    "inglés, en una o dos frases. Usa las herramientas para los pedidos, las "
    "existencias y los envíos; nunca inventes esos datos. Si una herramienta "
    "necesita datos que el cliente no ha dado, pídeselos. Otros datos de la "
    "tienda: 30 días para devolver un producto; 2 años de garantía; abierto de "
    "lunes a viernes de 10 a.m. a 8 p.m. y los sábados de 10 a.m. a 2 p.m.")

In [ ]:
print(json.dumps(SCHEMAS[0], ensure_ascii=False, indent=1))

## 4. The loop

In [ ]:
def generate(messages, schemas, max_tokens=200):
    """One answer from the model, with the tools described in the template."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    answer = tok.decode(output[0, inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"],
                          "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}, "text": block})
    return calls


def run(call):
    """Runs a call after checking its name and its arguments."""
    function = FUNCTIONS.get(call["name"])
    if function is None:
        return {"error": f"There is no tool called {call['name']}."}
    try:
        return function(**call["arguments"])
    except (TypeError, ValueError) as e:
        return {"error": f"Invalid arguments: {e}"}


def serve(request, schemas=SCHEMAS, system=SYSTEM_EN, max_steps=3):
    """Serves a request: the model can ask for tools up to max_steps times
    before giving the final answer."""
    messages = ([{"role": "system", "content": system}] if system else []) + [
        {"role": "user", "content": request}]
    log = []
    for _ in range(max_steps):
        answer = generate(messages, schemas)
        calls = read_calls(answer)
        if not calls:
            return answer, log
        messages.append({"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            result = run(call)
            log.append({**call, "result": result})
            messages.append({"role": "tool", "name": call["name"],
                             "content": json.dumps(result, ensure_ascii=False)})
    return "I couldn't complete your request; I'm passing it to the team.", log

One request with each system message. Watch whether the model calls the tool:

In [ ]:
request = "Where is my order NC-10871?"
for name, system in (("English", SYSTEM_EN), ("Spanish", SYSTEM_ES)):
    answer, log = serve(request, SCHEMAS, system)
    print(name, "→ calls:", [(r["name"], r["arguments"]) for r in log])
    print("   ", answer)

## 5. Twenty requests and four variants

In [ ]:
def number(expected):
    """Checks the order number, with or without the NC- prefix."""
    return lambda a: re.sub(r"\D", "", str(a.get("order_number", ""))) == expected[3:]


def product(expected, size=None):
    """Checks the product (name or reference) and, if needed, the size."""
    return lambda a: (strip_accents(a.get("product", "")) in
                      (strip_accents(expected[0]), strip_accents(expected[1]))
                      and (size is None or str(a.get("size", "")).upper() == size))


def destination(place):
    """Checks that the destination contains the expected place."""
    return lambda a: strip_accents(place) in strip_accents(a.get("destination", ""))


# Request, expected tool (or None), check of the arguments, and texts of which
# at least one must appear in the final answer.
REQUESTS = [
    ("Where is my order NC-10871?", "order_status", number("NC-10871"),
     ["shipped", "SS-558201"]),
    ("Hi, this is Oscar, order 11033. When will it arrive?", "order_status",
     number("NC-11033"), ["13"]),
    ("Has order nc-11107 shipped yet?", "order_status", number("NC-11107"),
     ["shipped", "has left", "on its way"]),
    ("My order NC-11302 is running late, what's going on?", "order_status",
     number("NC-11302"), ["size L", "stock", "15"]),
    ("When will I get my money back for order NC-10990?", "order_status",
     number("NC-10990"), ["14"]),
    ("What's happening with my order NC-99999?", "order_status",
     number("NC-99999"),
     ["no order", "not exist", "n't exist", "not found", "couldn't find",
      "could not find", "any order with"]),
    ("Where is my order?", None, None, ["number"]),
    ("Do you have the Aero 3 in size 54?", "stock",
     product(("Aero 3", "CR-7010"), "54"),
     ["October 20", "Oct 20", "2026-10-20", "10/20"]),
    ("Is there any City 5 left in size M?", "stock",
     product(("City 5", "UR-5150"), "M"), ["3", "three"]),
    ("Do you have reference EL-9050 in stock?", "stock",
     product(("E-City 600", "EL-9050")),
     ["November", "Nov 5", "2026-11-05", "11/5"]),
    ("Do you have the Mini Helmet in size S?", "stock",
     product(("Mini Helmet", "CA-4150"), "S"), ["no", "out of stock", "n't"]),
    ("How many E-Folding bikes do you have left?", "stock",
     product(("E-Folding", "EL-9200")), ["3", "three"]),
    ("Do you have the North Fixie?", "stock",
     product(("North Fixie", "UR-5470")), ["no ", "not ", "n't", "check"]),
    ("How much does it cost to ship a $30 order to Sacramento?", "shipping_cost",
     destination("Sacramento"), ["4.95"]),
    ("If I buy a $600 bike, will you ship it to Denver for free?",
     "shipping_cost", destination("Denver"), ["free", "no cost", "no charge"]),
    ("How much is shipping to Anchorage for an $80 order?", "shipping_cost",
     destination("Anchorage"), ["9.95"]),
    ("Do you ship to France?", "shipping_cost", destination("France"),
     ["quote", "no ", "not ", "n't"]),
    ("How long do I have to return a product?", None, None,
     ["30 days", "30-day"]),
    ("Hello, good morning.", None, None, None),
    ("Are you open on Saturdays?", None, None,
     ["10 a.m.", "10am", "10 am", "10:00", "2 p.m.", "2pm", "2 pm", "14:00"]),
]

In [ ]:
def evaluate(schemas=SCHEMAS, system=SYSTEM_EN):
    """Serves the twenty requests and checks the tool, the arguments and the
    answer."""
    rows = []
    for request, expected, arguments, keys in REQUESTS:
        answer, log = serve(request, schemas, system)
        used = [r["name"] for r in log]
        first = log[0] if log else None
        row = {"request": request, "answer": answer, "log": log,
               "tool": (used[:1] == [expected]) if expected else not used}
        row["arguments"] = (row["tool"] and expected is not None
                            and arguments(first["arguments"]))
        row["answer_ok"] = keys is None or any(
            k.lower() in answer.lower() for k in keys)
        rows.append(row)
    return rows


def summary(rows):
    """Tools chosen correctly (of 20), correct arguments (of 16), correct
    answers (of 20) and number of calls."""
    with_tool = [r for r, q in zip(rows, REQUESTS) if q[1]]
    return {"tool": sum(r["tool"] for r in rows),
            "arguments": sum(r["arguments"] for r in with_tool),
            "answer": sum(r["answer_ok"] for r in rows),
            "calls": sum(len(r["log"]) for r in rows)}


VARIANTS = {
    "Spanish system message": (SCHEMAS, SYSTEM_ES),
    "No system message": (SCHEMAS, None),
    "English system message": (SCHEMAS, SYSTEM_EN),
    "English system message, no descriptions": (BARE_SCHEMAS, SYSTEM_EN),
}

In [ ]:
results = {}
for name, (schemas, system) in VARIANTS.items():
    results[name] = evaluate(schemas, system)
    print(f"{name:40s}", summary(results[name]))

The requests of the English variant, with their calls and a mark on the ones that fail:

In [ ]:
for r in results["English system message"]:
    mark = "  " if r["tool"] and r["answer_ok"] else "✗ "
    print(mark + r["request"])
    print("      calls:", [(c["name"], c["arguments"]) for c in r["log"]])
    print("      answer:", r["answer"][:150])

## Exercises

1. Explain in your own words who does what in a function call: the model, the chat template and the program.

2. Print the full text the model receives, with `apply_chat_template` and the tools, for one of the requests. Where do the descriptions appear? What instructions does the template add?

3. For the requests in which the model does not use the tool with the English system message, try to improve the description of the corresponding tool. Can you get it to use the tool without breaking other requests?

4. Add a fourth tool, `search_product(description)`, that returns the catalog products whose description contains any word of the query. Does the choice of the other three change? Does the model get confused?

5. Change `order_status` so that it only accepts numbers in the exact format `NC-12345` and returns an error otherwise. How many requests fail now? Does the model correct the number after the error?

6. Write a request that needs two tools, such as "Do you have the City 5 in size M, and how much is shipping to Anchorage?". Does the model call both? In a single answer or in two steps?

7. Make the stock tool return a more explicit result, for example `"3 units"` instead of `3`. Is the E-Folding error fixed?

8. Translate the tool descriptions into Spanish as well, so that everything the model reads is in Spanish except the customer's request, and repeat the test with both models. Does it get worse? Which system message language suits the small model best?

9. Add a dangerous tool, `cancel_order(order_number)`, and change `run` so that, before running it, it asks for a confirmation from the keyboard with `input`. Try it with the request "Cancel my order NC-11033".

10. **Full assistant.** Combine the RAG of chapter 6 with the tools of this chapter: add a tool `search_documents(query)` that returns the three most relevant fragments. Try it with ten mixed requests and write a short report on what works and what does not.

## References

Models [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) and [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B). [Toolformer](https://arxiv.org/abs/2302.04761) (Schick et al., 2023). [Model Context Protocol](https://modelcontextprotocol.io). The North Cycles store, its orders and its warehouse are fictional. Original code and experiments.